# Train a small SongUNet prior

A minimal run of the training loop, then unconditional samples from the
result. With `DATA_DIR = None` the data are generated structured fields
(`data_source: structured`): smooth random fields that drift across the
patch, with known statistics, scaled so one model unit is 0.051 m of SSH.
They need no data files and give the model something real to learn. Point
`DATA_DIR` at the shared `training_patches/` directory (or your own data in
the same layout) to train on llc4320 SSH.

For real training use `scripts/train.py` on a GPU node; see the README.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

import obottle_starter as obs
from obottle_starter.training.data import build_dataset
from obottle_starter.training.train import train

ROOT = Path("..")
DATA_DIR = None  # e.g. ROOT / "training_patches"

cfg = OmegaConf.merge(
    OmegaConf.load(ROOT / "configs" / "train_base.yaml"),
    OmegaConf.load(ROOT / "configs" / "train_ssh_only.yaml"),
    {
        "output_dir": "runs/notebook_ssh",
        "time_length": 5,  # shorter windows train faster
        "model_channels": 32,  # ~10x fewer parameters than the release
        "batch_size": 8,
        "num_workers": 2,
        "num_steps": 500,
        "log_every": 50,
        "checkpoint_every": 500,
        "lr_rampup_steps": 100,
        "start": "auto",  # re-running the notebook continues the same run
    },
)
if DATA_DIR is None:
    cfg.data_source = "structured"  # generated fields, no files needed
    cfg.H = cfg.W = 32  # small patches: fast on a CPU
    cfg.channel_mult = [1, 2, 2]  # levels at 32, 16 and 8 cells
    cfg.attn_resolutions = [8]
    cfg.temporal_attn_resolutions = [8, 16]
else:
    cfg.data_source = "patches"
    cfg.data_dir = str(DATA_DIR)
    # the shared subset holds 2000 hours; hold out the last 150 for validation
    cfg.time_range_start, cfg.time_range_end = 0, 1500
    cfg.validation.time_range_start, cfg.validation.time_range_end = 1850, 2000
cfg.validation.every = 250  # held-out denoising loss by noise level
cfg.validation.n_samples = 16

## What the model sees

A sample is a normalized `(C, T, H*W)` target plus the time conditioning:
offsets of each frame from the first, local solar time and day of year.

In [ ]:
sample = build_dataset(cfg)[0]
print({k: tuple(v.shape) for k, v in sample.items()})
print("frame offsets (h):", sample["time_offsets"].tolist())
fig, axes = plt.subplots(1, cfg.time_length, figsize=(3 * cfg.time_length, 3))
for t, ax in enumerate(axes):
    ax.imshow(
        sample["target"][0, t].reshape(cfg.H, cfg.W), origin="lower", cmap="RdBu_r"
    )
    ax.set_title(f"+{sample['time_offsets'][t]:.0f} h")

In [ ]:
out_dir = train(cfg)
net, loaded = obs.load_checkpoint(
    out_dir / "checkpoints" / "checkpoint_latest.pt",
    "cuda" if torch.cuda.is_available() else "cpu",
)

## Sample the trained prior

`assimilate` with no masks draws prior samples for any set of frame times.

In [ ]:
times = np.datetime64("2012-02-01T00") + np.arange(loaded.time_length) * np.timedelta64(
    12, "h"
)
prior = obs.assimilate(
    net, loaded, times=times, center_lon=-150.0, masks={}, n_members=4, steps=32, seed=0
)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for m, ax in enumerate(axes):
    ax.imshow(prior.samples.sel(channel="SSH")[m, 0], origin="lower", cmap="RdBu_r")
    ax.set_title(f"member {m}")